# Weather Temperature Prediction using SimpleRNN

**Objective:** Use the previous 7 days of weather data (temperature, humidity, and wind speed) to predict the next day's temperature, evaluate the model, and forecast the next 7 days.

> **Dataset note:** The supplied weather dataset contains hourly observations. This notebook aggregates them into daily averages before creating sequences. Update `CSV_PATH` below if your CSV has a different filename or location.


## 1. Install and import libraries

Run the install cell only if these packages are not already installed in your environment.

In [ ]:
# If needed, uncomment and run:
# %pip install tensorflow pandas numpy matplotlib scikit-learn joblib

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, SimpleRNN, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow:", tf.__version__)


## 2. Load and understand the dataset

Make sure the CSV path points to your downloaded dataset.

In [ ]:
CSV_PATH = "dataset/weather.csv"  # Change this if your CSV filename is different

df = pd.read_csv(CSV_PATH)

print("First 10 rows:")
display(df.head(10))

print("\nDataset information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum())

print("\nColumns:")
print(df.columns.tolist())


## 3. Convert hourly observations into daily averages

The dataset uses `Formatted Date` and contains hourly weather readings. We extract the date and calculate daily averages for the required numerical features.

In [ ]:
required_columns = [
    "Formatted Date",
    "Temperature (C)",
    "Humidity",
    "Wind Speed (km/h)"
]

missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

df["Date"] = pd.to_datetime(
    df["Formatted Date"].astype(str).str[:10],
    errors="coerce"
)

hourly_df = df[[
    "Date",
    "Temperature (C)",
    "Humidity",
    "Wind Speed (km/h)"
]].copy()

hourly_df = hourly_df.dropna(subset=["Date"])

daily_df = (
    hourly_df.groupby("Date", as_index=False)
    [["Temperature (C)", "Humidity", "Wind Speed (km/h)"]]
    .mean()
    .sort_values("Date")
    .reset_index(drop=True)
)

daily_df = daily_df.dropna().reset_index(drop=True)

print("Daily dataset shape:", daily_df.shape)
display(daily_df.head(10))
print("\nMissing values after aggregation:")
display(daily_df.isnull().sum())


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(daily_df["Date"], daily_df["Temperature (C)"])
plt.title("Daily Temperature Trends")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 4. Scale features and create 7-day sequences

Each input sample contains **7 days × 3 features**. The target is the temperature on the following day.

To avoid data leakage, the scaler is fitted only on the training-period rows. The chronological split is 70% training, 15% validation, and 15% testing.

In [ ]:
FEATURES = [
    "Temperature (C)",
    "Humidity",
    "Wind Speed (km/h)"
]
TARGET_INDEX = 0
SEQUENCE_LENGTH = 7

values = daily_df[FEATURES].to_numpy(dtype=np.float32)
n_rows = len(values)

train_row_end = int(n_rows * 0.70)
val_row_end = int(n_rows * 0.85)

if train_row_end <= SEQUENCE_LENGTH:
    raise ValueError("Not enough daily rows to create 7-day sequences.")

scaler = MinMaxScaler()
scaler.fit(values[:train_row_end])  # Fit on training period only
scaled_values = scaler.transform(values)

X_all, y_all, target_indices = [], [], []

for i in range(SEQUENCE_LENGTH, n_rows):
    X_all.append(scaled_values[i-SEQUENCE_LENGTH:i, :])
    y_all.append(scaled_values[i, TARGET_INDEX])
    target_indices.append(i)

X_all = np.asarray(X_all, dtype=np.float32)
y_all = np.asarray(y_all, dtype=np.float32)
target_indices = np.asarray(target_indices)

train_mask = target_indices < train_row_end
val_mask = (target_indices >= train_row_end) & (target_indices < val_row_end)
test_mask = target_indices >= val_row_end

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val, y_val = X_all[val_mask], y_all[val_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]

print("All sequences:", X_all.shape)
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape, "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape, "y_test: ", y_test.shape)


## 5. Build the SimpleRNN model

The input shape is `(7, 3)`: seven time steps and three weather features. The Dense layer produces one continuous temperature value.

In [ ]:
model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, len(FEATURES))),
    SimpleRNN(64, activation="tanh"),
    Dropout(0.2),
    Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()


## 6. Train the model

Early stopping monitors validation loss and restores the best weights.

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("SimpleRNN Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.tight_layout()
plt.show()


## 7. Evaluate on the test set

The model predicts scaled temperatures. We convert both predictions and targets back to Celsius before calculating RMSE, MAE, and R².

In [ ]:
pred_scaled = model.predict(X_test, verbose=0).reshape(-1, 1)

# Inverse-transform only the temperature feature.
temp_min = scaler.data_min_[TARGET_INDEX]
temp_range = scaler.data_range_[TARGET_INDEX]

predicted_c = pred_scaled.ravel() * temp_range + temp_min
actual_c = y_test * temp_range + temp_min

rmse = np.sqrt(mean_squared_error(actual_c, predicted_c))
mae = mean_absolute_error(actual_c, predicted_c)
r2 = r2_score(actual_c, predicted_c)

print(f"RMSE: {rmse:.3f} °C")
print(f"MAE:  {mae:.3f} °C")
print(f"R²:   {r2:.4f}")


In [ ]:
test_dates = daily_df.loc[target_indices[test_mask], "Date"].reset_index(drop=True)

plt.figure(figsize=(12, 5))
plt.plot(test_dates, actual_c, label="Actual Temperature")
plt.plot(test_dates, predicted_c, label="Predicted Temperature")
plt.title("Actual vs Predicted Daily Temperature (Test Period)")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 8. Forecast the next 7 days

This is a recursive demonstration forecast. Since future humidity and wind speed are unknown, this simple approach carries forward their most recent scaled values. Therefore, interpret the forecast as an approximate model exercise, not an operational weather forecast.

In [ ]:
current_sequence = scaled_values[-SEQUENCE_LENGTH:].copy()
forecast_scaled = []

for _ in range(7):
    next_temp_scaled = float(
        model.predict(
            current_sequence.reshape(1, SEQUENCE_LENGTH, len(FEATURES)),
            verbose=0
        )[0, 0]
    )

    forecast_scaled.append(next_temp_scaled)

    # Carry forward the latest humidity and wind speed.
    next_day_features = current_sequence[-1].copy()
    next_day_features[TARGET_INDEX] = next_temp_scaled

    current_sequence = np.vstack([
        current_sequence[1:],
        next_day_features
    ])

forecast_c = np.asarray(forecast_scaled) * temp_range + temp_min
last_date = pd.to_datetime(daily_df["Date"].iloc[-1])
future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=7)

forecast_df = pd.DataFrame({
    "Date": future_dates,
    "Predicted Temperature (C)": forecast_c
})

display(forecast_df)


In [ ]:
recent_df = daily_df.tail(30)

plt.figure(figsize=(12, 5))
plt.plot(
    recent_df["Date"],
    recent_df["Temperature (C)"],
    label="Recent Historical Temperature"
)
plt.plot(
    future_dates,
    forecast_c,
    marker="o",
    linestyle="--",
    label="Next 7 Days Forecast"
)
plt.title("7-Day Temperature Forecast")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 9. Save project outputs

This saves the trained model, fitted scaler, and forecast CSV so they can be reused outside the notebook.

In [ ]:
os.makedirs("outputs", exist_ok=True)

model.save("outputs/weather_rnn.keras")

import joblib
joblib.dump(scaler, "outputs/weather_scaler.pkl")

forecast_df.to_csv("outputs/forecast_results.csv", index=False)

print("Saved:")
print("- outputs/weather_rnn.keras")
print("- outputs/weather_scaler.pkl")
print("- outputs/forecast_results.csv")


## 10. Conclusion

In this project, we:
- Explored hourly weather observations and aggregated them into daily averages.
- Normalized temperature, humidity, and wind speed.
- Created sequences from the previous 7 days to predict the next day's temperature.
- Built and trained a SimpleRNN regression model.
- Evaluated test predictions using RMSE, MAE, and R².
- Generated an approximate 7-day recursive forecast.

**Remember:** Results depend on the dataset, chronological split, preprocessing, and model training. Report the metrics generated by running this notebook rather than copying results from a different run.
